# Notebook 2: Subcluster Annotation

This notebook performs fine-grained subclustering and annotation of Stromal/CAF, T cell, and Myeloid populations from the BICRC Xenium dataset.

**Overview:**
1. Stromal/CAF subclustering: GPU-accelerated clustering → 17 subclusters (c0–c16)
2. T cell subclustering: 11 subclusters (c0–c10)
3. Myeloid subclustering: 13 subclusters (c0–c12)
4. Differential expression (Wilcoxon rank-sum) for each subpopulation
5. Leiden-based annotation with biologically meaningful labels
6. UMAP and dot plot visualization

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import rapids_singlecell as rsc
import cupy as cp
import anndata

sc.settings.verbosity = 3
sc.settings.figdir = '../figures/'
plt.rcParams['figure.dpi'] = 150

## 2. Load Data

In [ ]:
# Load pre-computed subcluster h5ad files
adata_caf = sc.read_h5ad('../data/integrate_adata_filtered_caf.h5ad')
adata_tcell = sc.read_h5ad('../data/integrate_adata_filtered_tcell.h5ad')
adata_mye = sc.read_h5ad('../data/integrate_adata_filtered_mye.h5ad')

print('CAF:', adata_caf)
print('T cell:', adata_tcell)
print('Myeloid:', adata_mye)

## 3. GPU-Accelerated Subclustering

rapids_singlecell is used for GPU-accelerated PCA, neighborhood graph construction, UMAP, and Leiden clustering.

In [ ]:
def run_gpu_clustering(adata, resolution=0.5, random_state=0, prefix='c'):
    """Run full clustering pipeline on GPU using rapids_singlecell."""
    rsc.get.anndata_to_GPU(adata)
    rsc.tl.pca(adata)
    rsc.pp.neighbors(adata, random_state=random_state)
    rsc.tl.umap(adata, random_state=random_state)
    rsc.tl.leiden(adata, resolution=resolution, random_state=random_state)
    # Transfer back to CPU for downstream analysis
    rsc.get.anndata_to_CPU(adata)
    return adata

### 3.1 Stromal/CAF Subclustering

In [ ]:
adata_caf = run_gpu_clustering(adata_caf, resolution=0.5)
adata_caf.obs['leiden'] = ['Stromal ' + c for c in adata_caf.obs['leiden']]
print('Stromal leiden clusters:', sorted(adata_caf.obs['leiden'].unique()))

### 3.2 T Cell Subclustering

In [ ]:
adata_tcell = run_gpu_clustering(adata_tcell, resolution=0.5)
adata_tcell.obs['leiden'] = ['T cell ' + c for c in adata_tcell.obs['leiden']]
print('T cell leiden clusters:', sorted(adata_tcell.obs['leiden'].unique()))

### 3.3 Myeloid Subclustering

In [ ]:
adata_mye = run_gpu_clustering(adata_mye, resolution=0.5)
adata_mye.obs['leiden'] = ['Myeloid ' + c for c in adata_mye.obs['leiden']]
print('Myeloid leiden clusters:', sorted(adata_mye.obs['leiden'].unique()))

## 4. Differential Expression Analysis (Wilcoxon Rank-Sum)

Marker genes are identified for each subcluster using the Wilcoxon rank-sum test.

In [ ]:
def run_deg_analysis(adata, groupby='leiden', n_genes=20):
    """Run Wilcoxon DEG analysis and return ranked genes dataframe."""
    sc.tl.rank_genes_groups(adata, groupby=groupby, method='wilcoxon', pts=True)
    result = sc.get.rank_genes_groups_df(adata, group=None)
    return result

deg_caf = run_deg_analysis(adata_caf)
deg_tcell = run_deg_analysis(adata_tcell)
deg_mye = run_deg_analysis(adata_mye)

# Save DEG results
deg_caf.to_csv('../results/deg_stromal_subclusters.csv', index=False)
deg_tcell.to_csv('../results/deg_tcell_subclusters.csv', index=False)
deg_mye.to_csv('../results/deg_myeloid_subclusters.csv', index=False)

## 5. Leiden Cluster Annotation

Clusters are annotated with biologically meaningful labels based on marker gene expression and spatial context.

### 5.1 Stromal/CAF Annotation (17 clusters)

In [ ]:
# Stromal subcluster annotation based on marker genes and spatial context
caf_annotation = {
    'Stromal c0':  'c0_Epi_CAF_boundary',
    'Stromal c1':  'c1_POSTN+_CAF',
    'Stromal c2':  'c2_Endo_CAF_boundary',
    'Stromal c3':  'c3_POSTN+_CAF',
    'Stromal c4':  'c4_MMP11+_CAF',
    'Stromal c5':  'c5_ACTA2+_myCAF',
    'Stromal c6':  'c6_CXCL14+_CAF',
    'Stromal c7':  'c7_ADH1B+_CAF',
    'Stromal c8':  'c8_PDPN+_CAF',
    'Stromal c9':  'c9_RGS5+_pericyte',
    'Stromal c10': 'c10_PECAM1+_Endo',
    'Stromal c11': 'c11_ACKR1+_Endo',
    'Stromal c12': 'c12_Mye_Str_boundary',
    'Stromal c13': 'c13_T_Str_boundary',
    'Stromal c14': 'c14_B_Str_boundary',
    'Stromal c15': 'c15_Smooth_muscle',
    'Stromal c16': 'c16_DC_Str_boundary',
}

adata_caf.obs['subcluster'] = adata_caf.obs['leiden'].map(caf_annotation)
print('CAF subclusters:', adata_caf.obs['subcluster'].value_counts())

### 5.2 T Cell Annotation (11 clusters)

In [ ]:
# T cell subcluster annotation
tcell_annotation = {
    'T cell c0':  'c0_KLRG1+_CD8_T',
    'T cell c1':  'c1_Treg',
    'T cell c2':  'c2_GZMA+_CD8_T',
    'T cell c3':  'c3_Str_T_boundary',
    'T cell c4':  'c4_B_T_boundary',
    'T cell c5':  'c5_Str_T_boundary',
    'T cell c6':  'c6_Epi_T_boundary',
    'T cell c7':  'c7_BAG3+_CD8_T',
    'T cell c8':  'c8_DC_T_boundary',
    'T cell c9':  'c9_CXCL13+_CD4_T',
    'T cell c10': 'c10_HDC+_Mast',
}

adata_tcell.obs['subcluster'] = adata_tcell.obs['leiden'].map(tcell_annotation)
print('T cell subclusters:', adata_tcell.obs['subcluster'].value_counts())

### 5.3 Myeloid Annotation (13 clusters)

In [ ]:
# Myeloid subcluster annotation
myeloid_annotation = {
    'Myeloid c0':  'c0_F13A1+_MΦ',
    'Myeloid c1':  'c1_Epi_Mye_boundary',
    'Myeloid c2':  'c2_CD1C+_DC',
    'Myeloid c3':  'c3_Epi_Mye_boundary',
    'Myeloid c4':  'c4_THBS+_Mye',
    'Myeloid c5':  'c5_CAF_Mye_boundary',
    'Myeloid c6':  'c6_T_Mye_boundary',
    'Myeloid c7':  'c7_OLR1+_Mye',
    'Myeloid c8':  'c8_CHIT1+_Mye',
    'Myeloid c9':  'c9_CXCL5+_DC',
    'Myeloid c10': 'c10_ADAMDEC1+_Mye',
    'Myeloid c11': 'c11_Lym_Mye_boundary',
    'Myeloid c12': 'c12_IL11+_MΦ',
}

adata_mye.obs['subcluster'] = adata_mye.obs['leiden'].map(myeloid_annotation)
print('Myeloid subclusters:', adata_mye.obs['subcluster'].value_counts())

## 6. UMAP Visualization

In [ ]:
def plot_subcluster_umap(adata, color_key, title, output_path):
    """Plot UMAP colored by subcluster annotation."""
    fig, ax = plt.subplots(figsize=(10, 8))
    sc.pl.umap(adata, color=color_key, ax=ax, show=False, title=title,
               legend_loc='right margin', legend_fontsize=8)
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()

plot_subcluster_umap(
    adata_caf, 'subcluster',
    'Stromal/CAF Subclusters',
    '../figures/02_umap_stromal_subclusters.png'
)
plot_subcluster_umap(
    adata_tcell, 'subcluster',
    'T Cell Subclusters',
    '../figures/02_umap_tcell_subclusters.png'
)
plot_subcluster_umap(
    adata_mye, 'subcluster',
    'Myeloid Subclusters',
    '../figures/02_umap_myeloid_subclusters.png'
)

## 7. Marker Gene Dot Plots

In [ ]:
# Key marker genes for stromal subclusters
caf_markers = [
    'POSTN', 'MMP11', 'ACTA2', 'CXCL14', 'ADH1B', 'PDPN',
    'RGS5', 'PECAM1', 'ACKR1', 'F13A1', 'CXCL5', 'DCN'
]

sc.pl.dotplot(
    adata_caf, caf_markers, groupby='subcluster',
    dendrogram=False, show=False
)
plt.savefig('../figures/02_dotplot_stromal_markers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Key marker genes for T cell subclusters
tcell_markers = [
    'KLRG1', 'CD8A', 'FOXP3', 'GZMA', 'GZMB', 'BAG3',
    'CXCL13', 'CD4', 'HDC', 'TPSAB1', 'IDO1', 'TIGIT'
]

sc.pl.dotplot(
    adata_tcell, tcell_markers, groupby='subcluster',
    dendrogram=False, show=False
)
plt.savefig('../figures/02_dotplot_tcell_markers.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Key marker genes for myeloid subclusters
myeloid_markers = [
    'F13A1', 'CD1C', 'THBS1', 'OLR1', 'CHIT1', 'CXCL5',
    'ADAMDEC1', 'IL11', 'CD68', 'ITGAX', 'S100A8', 'MRC1'
]

sc.pl.dotplot(
    adata_mye, myeloid_markers, groupby='subcluster',
    dendrogram=False, show=False
)
plt.savefig('../figures/02_dotplot_myeloid_markers.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Save Annotated Subcluster Objects

## 8. Add leiden_annotated Column

The `leiden_annotated` column is a copy of `subcluster` maintained for compatibility with downstream notebooks and the original analysis pipeline.

In [ ]:
# Create leiden_annotated column as alias for subcluster
# (used by downstream notebooks for backward compatibility)
for adata_obj in [adata_caf, adata_tcell, adata_mye]:
    adata_obj.obs['leiden_annotated'] = adata_obj.obs['subcluster']

print('leiden_annotated added to all objects')

In [ ]:
adata_caf.write_h5ad('../data/integrate_adata_filtered_caf_annotated.h5ad')
adata_tcell.write_h5ad('../data/integrate_adata_filtered_tcell_annotated.h5ad')
adata_mye.write_h5ad('../data/integrate_adata_filtered_mye_annotated.h5ad')

print('Saved all annotated subcluster objects.')